In [1]:
from pathlib import Path
import sys
import numpy as np

BASE_DIR = Path().cwd().parent
PICTURE_DATA_DIR = BASE_DIR / "picture_data"
DATASET_DIR = PICTURE_DATA_DIR / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
VALIDATION_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

AUGMENTED_DIR = PICTURE_DATA_DIR / "augmented"

FINE_TUNING_DIR = BASE_DIR / "fine_tuning"
FEATURE_DIR = FINE_TUNING_DIR / "features"

CLASS_NAMES = [
    "brille",
    "flasche",
    "geldboerse",
    "handy",
    "hund",
    "kirche",
    "schuh",
    "tasse"
]

VERSION = 2

sys.path.append(str(BASE_DIR))

from data_handling.manage_data import create_dataset, extract_features
from model.build_model import build_feature_extractor, split_base_model

In [2]:
train_ds = create_dataset(TRAIN_DIR, CLASS_NAMES, batch_size=32)
validation_ds = create_dataset(VALIDATION_DIR, CLASS_NAMES, batch_size=16)
test_ds = create_dataset(TEST_DIR, CLASS_NAMES, batch_size=16)

train_ds_augmented = create_dataset(AUGMENTED_DIR, CLASS_NAMES, batch_size=32)

train_ds = train_ds.concatenate(train_ds_augmented)

Found 1270 files belonging to 8 classes.
Found 596 files belonging to 8 classes.
Found 369 files belonging to 8 classes.
Found 3810 files belonging to 8 classes.


In [3]:
feature_extractor, _ = split_base_model()

In [4]:
X_train, y_train = extract_features(feature_extractor, train_ds)
X_validation, y_validation = extract_features(feature_extractor, validation_ds)
X_test, y_test = extract_features(feature_extractor, test_ds)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    FEATURE_DIR / f"features_{VERSION}.npz",
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    X_test=X_test,
    y_test=y_test
)